In [1]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\steve\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\steve\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\steve\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\steve\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\steve\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\steve\AppData\Roaming\nltk_data...


True

In [3]:
df = pd.read_csv("../data/raw/customer_feedback.csv")

print("Dataset loaded successfully!")
print("Total records:", len(df))

df.head()

Dataset loaded successfully!
Total records: 10000


,label,title,content
0,0,Defected Product,I purchased this product because my cat prefer...
1,0,problem with product,The product I received came in time but was da...
2,0,can't handle power losses - active ports fry o...,I have just fried my second of this model. Whe...
3,0,"Looks great, poor performance",I bought this high end kettle thinking I would...
4,1,perfect for loose teas,Looked a long time for this. Careful- it holds...


In [4]:
df['review_text'] = (
    df['title'].fillna('') + ". " +
    df['content'].fillna('')
)

df[['title', 'content', 'review_text']].head()

,title,content,review_text
0,Defected Product,I purchased this product because my cat prefer...,Defected Product. I purchased this product bec...
1,problem with product,The product I received came in time but was da...,problem with product. The product I received c...
2,can't handle power losses - active ports fry o...,I have just fried my second of this model. Whe...,can't handle power losses - active ports fry o...
3,"Looks great, poor performance",I bought this high end kettle thinking I would...,"Looks great, poor performance. I bought this h..."
4,perfect for loose teas,Looked a long time for this. Careful- it holds...,perfect for loose teas. Looked a long time for...


In [5]:
df['lowercase_text'] = df['review_text'].str.lower()

print("Original:")
print(df['review_text'].iloc[0])

print("\nLowercase:")
print(df['lowercase_text'].iloc[0])

Original:
Defected Product. I purchased this product because my cat prefers to drink from running water source. I bought it to replace a fountain we had for three years. This one did not even last a few months. At first I was pleased with the noise level (nice and quiet) but within a month the water stopped shooting out of the mouth. I believe the motor is defected. The only way it works if you remove the fish and with cover. Not sure if this is an isolated case but I would be wary of purchasing the Cats Rule Fountains. Too bad because they are cute!

Lowercase:
defected product. i purchased this product because my cat prefers to drink from running water source. i bought it to replace a fountain we had for three years. this one did not even last a few months. at first i was pleased with the noise level (nice and quiet) but within a month the water stopped shooting out of the mouth. i believe the motor is defected. the only way it works if you remove the fish and with cover. not sure if

In [6]:
def clean_text(text):

    text = str(text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


df['cleaned_text'] = df['lowercase_text'].apply(clean_text)

print("Before cleaning:")
print(df['lowercase_text'].iloc[0])

print("\nAfter cleaning:")
print(df['cleaned_text'].iloc[0])

Before cleaning:
defected product. i purchased this product because my cat prefers to drink from running water source. i bought it to replace a fountain we had for three years. this one did not even last a few months. at first i was pleased with the noise level (nice and quiet) but within a month the water stopped shooting out of the mouth. i believe the motor is defected. the only way it works if you remove the fish and with cover. not sure if this is an isolated case but i would be wary of purchasing the cats rule fountains. too bad because they are cute!

After cleaning:
defected product i purchased this product because my cat prefers to drink from running water source i bought it to replace a fountain we had for three years this one did not even last a few months at first i was pleased with the noise level nice and quiet but within a month the water stopped shooting out of the mouth i believe the motor is defected the only way it works if you remove the fish and with cover not sure

In [7]:
df['tokens'] = df['cleaned_text'].apply(word_tokenize)

print("Original text:")
print(df['cleaned_text'].iloc[0])

print("\nTokens:")
print(df['tokens'].iloc[0])

Original text:
defected product i purchased this product because my cat prefers to drink from running water source i bought it to replace a fountain we had for three years this one did not even last a few months at first i was pleased with the noise level nice and quiet but within a month the water stopped shooting out of the mouth i believe the motor is defected the only way it works if you remove the fish and with cover not sure if this is an isolated case but i would be wary of purchasing the cats rule fountains too bad because they are cute

Tokens:
['defected', 'product', 'i', 'purchased', 'this', 'product', 'because', 'my', 'cat', 'prefers', 'to', 'drink', 'from', 'running', 'water', 'source', 'i', 'bought', 'it', 'to', 'replace', 'a', 'fountain', 'we', 'had', 'for', 'three', 'years', 'this', 'one', 'did', 'not', 'even', 'last', 'a', 'few', 'months', 'at', 'first', 'i', 'was', 'pleased', 'with', 'the', 'noise', 'level', 'nice', 'and', 'quiet', 'but', 'within', 'a', 'month', 'the'

In [8]:
stop_words = set(stopwords.words('english'))

df['filtered_tokens'] = df['tokens'].apply(
    lambda tokens: [
        word for word in tokens
        if word not in stop_words
    ]
)

print("Before stop-word removal:")
print(df['tokens'].iloc[0])

print("\nAfter stop-word removal:")
print(df['filtered_tokens'].iloc[0])

Before stop-word removal:
['defected', 'product', 'i', 'purchased', 'this', 'product', 'because', 'my', 'cat', 'prefers', 'to', 'drink', 'from', 'running', 'water', 'source', 'i', 'bought', 'it', 'to', 'replace', 'a', 'fountain', 'we', 'had', 'for', 'three', 'years', 'this', 'one', 'did', 'not', 'even', 'last', 'a', 'few', 'months', 'at', 'first', 'i', 'was', 'pleased', 'with', 'the', 'noise', 'level', 'nice', 'and', 'quiet', 'but', 'within', 'a', 'month', 'the', 'water', 'stopped', 'shooting', 'out', 'of', 'the', 'mouth', 'i', 'believe', 'the', 'motor', 'is', 'defected', 'the', 'only', 'way', 'it', 'works', 'if', 'you', 'remove', 'the', 'fish', 'and', 'with', 'cover', 'not', 'sure', 'if', 'this', 'is', 'an', 'isolated', 'case', 'but', 'i', 'would', 'be', 'wary', 'of', 'purchasing', 'the', 'cats', 'rule', 'fountains', 'too', 'bad', 'because', 'they', 'are', 'cute']

After stop-word removal:
['defected', 'product', 'purchased', 'product', 'cat', 'prefers', 'drink', 'running', 'water', '

In [9]:
lemmatizer = WordNetLemmatizer()

df['lemmatized_tokens'] = df['filtered_tokens'].apply(
    lambda tokens: [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]
)

print("Before lemmatization:")
print(df['filtered_tokens'].iloc[0])

print("\nAfter lemmatization:")
print(df['lemmatized_tokens'].iloc[0])

Before lemmatization:
['defected', 'product', 'purchased', 'product', 'cat', 'prefers', 'drink', 'running', 'water', 'source', 'bought', 'replace', 'fountain', 'three', 'years', 'one', 'even', 'last', 'months', 'first', 'pleased', 'noise', 'level', 'nice', 'quiet', 'within', 'month', 'water', 'stopped', 'shooting', 'mouth', 'believe', 'motor', 'defected', 'way', 'works', 'remove', 'fish', 'cover', 'sure', 'isolated', 'case', 'would', 'wary', 'purchasing', 'cats', 'rule', 'fountains', 'bad', 'cute']

After lemmatization:
['defected', 'product', 'purchased', 'product', 'cat', 'prefers', 'drink', 'running', 'water', 'source', 'bought', 'replace', 'fountain', 'three', 'year', 'one', 'even', 'last', 'month', 'first', 'pleased', 'noise', 'level', 'nice', 'quiet', 'within', 'month', 'water', 'stopped', 'shooting', 'mouth', 'believe', 'motor', 'defected', 'way', 'work', 'remove', 'fish', 'cover', 'sure', 'isolated', 'case', 'would', 'wary', 'purchasing', 'cat', 'rule', 'fountain', 'bad', 'cute

In [10]:
import os

os.makedirs("../data/processed", exist_ok=True)

df.to_csv(
    "../data/processed/customer_feedback_processed.csv",
    index=False
)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!
